In [31]:
import csv
import json
import os # operating system library, let's us do things like making new folders
import requests
from time import sleep
from bs4 import BeautifulSoup # the scraping library!


SCRAPED_PAGES_DIRECTORY = "scraped_pages/"
SCRAPED_PAGES_PREFIX = "page"
scraped_pages_count = 0 # keep track of how many pages we have scraped

headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    global scraped_pages_count  # use the scraped_pages_count from the top of the script, instead of making a new local one
    filename = f"{SCRAPED_PAGES_DIRECTORY}{SCRAPED_PAGES_PREFIX}{scraped_pages_count}.html"
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        scraped_pages_count = scraped_pages_count + 1
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)
        scraped_pages_count = scraped_pages_count + 1
    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

url = "https://mapping-museums.bbk.ac.uk/2026/09/29/ten-years-of-mapping-museums/"  # URL we want to scrape

html = fetch_page(url)  # grabs the URL using our fetch function
soup = BeautifulSoup(html, "html.parser")  # soups it into a usable object

print(len(html)) # length of characters in HTML  

156046


In [32]:
entry = soup.find(class_="entry-content")

children = entry.find_all(True, recursive=False)  # True means find all elements, recursive means only find the top-level tags of the direct children and not any children thereof)

for index, child in enumerate(children):  # enumerate gives us both the contents of the list item, as well as its index number
    print(index, child.name, "|", child.get_text(" ", strip=True))  # print the index number, the type of element, and the text contents of each child element

0 p | It is ten years ago this month that the first Mapping Museums project started. In September 2016, Alexandra Poulovassilis, Jamie Larkin, and I began the process of collecting, modelling, and analysing data on the UK museums sector since 1960. Andrea Ballatore soon joined, bringing expertise in geography and data analysis. Since then, the team has grown, and we’ve completed three major projects. I want to take a moment to think about what we’ve achieved and what is left to do.
1 p | The Mapping Museums database showed that the UK museum sector was bigger and more varied than assumed, and it made non-national museums, and especially local authority and independent museums more visible within the sector than had previously been the case. It was also the first time that non-accredited museums had been documented. When the database first launched in 2020, Lisa Ollerhead (Senior policy advisor for museums at the Department of Digital, Culture, Media and Sport and now director of the As

In [34]:
url = "https://mapping-museums.bbk.ac.uk/2026/09/22/archiving-websites-about-closed-museums/"  # URL we want to scrape

html = fetch_page(url)  # grabs the URL using our fetch function
soup = BeautifulSoup(html, "html.parser")  # soups it into a usable object

print(len(html)) # length of characters in HTML  

entry = soup.find(class_="entry-content")



152391


In [40]:
children = entry.find_all(True, recursive=False)  # True means find all elements, recursive means only find the top-level tags of the direct children and not any children thereof)
entry_paragraphs = []

for index, child in enumerate(children):  # enumerate gives us both the contents of the list item, as well as its index number
    # print(index, child.name, "|", child.get_text(" ", strip=True))  # print the index number, the type of element, and the text contents of each child element
    pg = child.get_text(" ", strip=True)
    if len(pg) > 0:  # change to 30 to remove name from set
        entry_paragraphs.append(pg)

print(entry_paragraphs)
print(len(entry_paragraphs))

['When we launched our report on museum closure last December, the British Library approached us to ask if we would be interested in using our research as the basis for a website archive.', 'It seemed a very appropriate project to get involved in. Websites have often proved to be invaluable sources for information about museums – not only for those that are open, of course, but also for those that have closed. Online resources have provided data for all three projects that the Mapping Museums Lab has undertaken, but our most recent project on museum closure highlighted once again a problem that’s well known by those that undertake research online.', 'For our first project, Mapping Museums, we recorded the sources of information for each museum, which frequently included websites. When we checked some of those again for our latest project six years later, many of those sites had gone offline. This phenomenon, known as ‘link rot’, is all too common and services have been developed to dea

In [42]:
# making a function that does all of the above and returns the data

def scrape_data(html, url):
    soup = BeautifulSoup(html, "html.parser")  # soups it into a usable object
    entry_content = soup.find(class_="entry-content")
    children = entry_content.find_all(True, recursive=False)  # True means find all elements, recursive means only find the top-level tags of the direct children and not any children thereof)
    entry = {
        "url": url,
        "paragraphs": []
    }
    
    for index, child in enumerate(children):  # enumerate gives us both the contents of the list item, as well as its index number
        pg = child.get_text(" ", strip=True)
        if len(pg) > 0:  # change to 30 to remove name from set
            entry["paragraphs"].append(pg)

    return entry



In [43]:
# use the function

url = "https://mapping-museums.bbk.ac.uk/2026/03/03/major-changes-to-the-mapping-museums-lab-database/"  # URL we want to scrape
html = fetch_page(url)

print(scrape_data(html, url))

{'url': 'https://mapping-museums.bbk.ac.uk/2026/03/03/major-changes-to-the-mapping-museums-lab-database/', 'paragraphs': ['We’re delighted to announce that key functions from the original Mapping Museums Lab database have now been fully integrated into our new web application .', 'We are very grateful to Art Fund for helping fund additional development work, which has enabled us to extend and strengthen the Mapping Museums Lab resource for the benefit of the wider museums and heritage community.', 'This update brings together, in one place, the rich data from our first major project — a comprehensive database of all museums open in the UK from 1960 to the present — with our second major strand of research: detailed information on museums that have closed, including reasons for closure and the outcomes for their collections. For the first time, users can seamlessly explore all the Mapping Museums Lab data in one site.', 'Responding to input from our sector evaluation sessions, the addit

In [45]:
# use the function in a for loop with a list of URLs

scraped_pages_count = 0 # resetting the scraped pages count for this example

urls = [
    "https://mapping-museums.bbk.ac.uk/2026/01/28/new-article-on-museum-closure-data/",
    "https://mapping-museums.bbk.ac.uk/2026/03/03/major-changes-to-the-mapping-museums-lab-database/",
    "https://mapping-museums.bbk.ac.uk/2020/03/17/the-mapping-museums-website-and-database-is-now-live/"
]

blog_data = []

for url in urls:
    html = fetch_page(url)
    entry_data = scrape_data(html,url)
    blog_data.append(entry_data)

print(blog_data)

[{'url': 'https://mapping-museums.bbk.ac.uk/2026/01/28/new-article-on-museum-closure-data/', 'paragraphs': ['The Mapping Museums Lab has just published an article in the Journal of Open Humanities Data about our project on museum closure. If you are interested in finding out more about how we collected and managed the data, the article is open access and free for everyone to read: A Dataset of Collections Dispersal Following Museum Closures in the UK During 2000–2025 .']}, {'url': 'https://mapping-museums.bbk.ac.uk/2026/03/03/major-changes-to-the-mapping-museums-lab-database/', 'paragraphs': ['We’re delighted to announce that key functions from the original Mapping Museums Lab database have now been fully integrated into our new web application .', 'We are very grateful to Art Fund for helping fund additional development work, which has enabled us to extend and strengthen the Mapping Museums Lab resource for the benefit of the wider museums and heritage community.', 'This update brings

In [46]:
len(blog_data)

3

In [47]:
# save as JSON

with open("entries.json", "w", encoding="utf-8") as file:
    json.dump(blog_data, file, indent=4, ensure_ascii=False)
